## Step 13: KPI Computation & Business Metrics

---

### Purpose

This notebook computes **final business KPIs** from validated and trusted data.
All metrics are derived from standardized definitions and are suitable for
automated daily and weekly reporting.

KPIs computed:
- Revenue
- Orders
- Average Order Value (AOV)
- Week-over-Week (WoW) Growth
- Month-over-Month (MoM) Growth
- Platform and Location breakdowns

---


In [1]:
import pandas as pd
import numpy as np

base_orders = pd.read_csv(
    "/content/drive/MyDrive/ML_PROJECT/Automated Business Reporting/data/base_orders_cleaned.csv",
    parse_dates=["order_date"]
)

base_orders.head()


,order_date,order_id,year,month,week,customer_id,platform_id,platform_name,location_id,location_name,region,order_amount,is_negative_revenue,is_future_date,is_missing_key,is_duplicate_order,is_anomalous_day_x,is_anomalous_day_y,is_anomalous_day
0,2024-01-28,1,2024,1,4,846,1,Web,1,Delhi,North,3022.67,False,False,False,False,False,False,False
1,2024-09-27,2,2024,9,39,1056,3,Marketplace,3,Bangalore,South,1396.24,False,False,False,False,False,False,False
2,2024-01-20,3,2024,1,3,433,3,Marketplace,2,Mumbai,West,485.25,False,False,False,False,False,False,False
3,2025-12-28,4,2025,12,52,537,1,Web,4,Hyderabad,South,2753.72,False,False,False,False,False,False,False
4,2025-12-19,5,2025,12,51,66,2,Mobile App,2,Mumbai,West,1997.41,False,False,False,False,False,False,False


In [2]:
base_orders["date"] = base_orders["order_date"].dt.date
base_orders["week"] = base_orders["order_date"].dt.isocalendar().week
base_orders["year"] = base_orders["order_date"].dt.year
base_orders["month"] = base_orders["order_date"].dt.to_period("M")


In [3]:
# DAILY KPIs (Core Metrics)

In [4]:
## Daily Revenue, Orders, AOV

daily_kpis = (
    base_orders
    .groupby("date")
    .agg(
        revenue=("order_amount", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

daily_kpis["aov"] = daily_kpis["revenue"] / daily_kpis["orders"]
daily_kpis.head()


,date,revenue,orders,aov
0,2024-01-01,96142.81,63,1526.076349
1,2024-01-02,108560.79,56,1938.585536
2,2024-01-03,77601.77,50,1552.035400
3,2024-01-04,103105.12,64,1611.017500
4,2024-01-05,70043.90,47,1490.295745


In [5]:
# WEEKLY KPIs & WoW Growth

In [6]:
## Weekly Aggregation

weekly_kpis = (
    base_orders
    .groupby(["year", "week"])
    .agg(
        revenue=("order_amount", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
    .sort_values(["year", "week"])
)

weekly_kpis["aov"] = weekly_kpis["revenue"] / weekly_kpis["orders"]
weekly_kpis.head()


,year,week,revenue,orders,aov
0,2024,1,801617.56,508,1577.987323
1,2024,2,692341.23,450,1538.536067
2,2024,3,769282.40,472,1629.835593
3,2024,4,617181.99,382,1615.659660
4,2024,5,708509.73,437,1621.303730


In [7]:
# Week-over-Week Growth

weekly_kpis["revenue_wow_pct"] = weekly_kpis["revenue"].pct_change() * 100
weekly_kpis["orders_wow_pct"] = weekly_kpis["orders"].pct_change() * 100


In [8]:
# MONTHLY KPIs & MoM Growth

## Monthly Aggregation

monthly_kpis = (
    base_orders
    .groupby("month")
    .agg(
        revenue=("order_amount", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
    .sort_values("month")
)

monthly_kpis["aov"] = monthly_kpis["revenue"] / monthly_kpis["orders"]
monthly_kpis.head()


,month,revenue,orders,aov
0,2024-01,2987485.44,1879,1589.933709
1,2024-02,2888480.67,1775,1627.313054
2,2024-03,3012870.25,1856,1623.313712
3,2024-04,3059430.31,1896,1613.623581
4,2024-05,2969446.84,1832,1620.877096


In [9]:
# Month-over-Month Growth

monthly_kpis["revenue_mom_pct"] = monthly_kpis["revenue"].pct_change() * 100
monthly_kpis["orders_mom_pct"] = monthly_kpis["orders"].pct_change() * 100


In [10]:
# PLATFORM-LEVEL KPIs (Business-Critical)

platform_kpis = (
    base_orders
    .groupby("platform_name")
    .agg(
        revenue=("order_amount", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

platform_kpis["aov"] = platform_kpis["revenue"] / platform_kpis["orders"]
platform_kpis.sort_values("revenue", ascending=False)


,platform_name,revenue,orders,aov
2,Web,36102820.98,22368,1614.038849
1,Mobile App,21623229.34,13599,1590.060250
0,Marketplace,14388734.62,8981,1602.130567


In [11]:
# LOCATION-LEVEL KPIs

location_kpis = (
    base_orders
    .groupby("location_name")
    .agg(
        revenue=("order_amount", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

location_kpis["aov"] = location_kpis["revenue"] / location_kpis["orders"]
location_kpis.sort_values("revenue", ascending=False)


,location_name,revenue,orders,aov
3,Mumbai,18242499.70,11238,1623.287035
1,Delhi,18139528.87,11259,1611.113675
0,Bangalore,17989589.33,11276,1595.387489
2,Hyderabad,17743167.04,11175,1587.755440


In [12]:
# KPI Sanity Checks

assert (daily_kpis["revenue"] >= 0).all()
assert (weekly_kpis["orders"] > 0).all()
assert (monthly_kpis["aov"] > 0).all()


In [13]:
daily_kpis.to_csv("/content/drive/MyDrive/ML_PROJECT/Automated Business Reporting/data/daily_kpis.csv", index=False)
weekly_kpis.to_csv("/content/drive/MyDrive/ML_PROJECT/Automated Business Reporting/data/weekly_kpis.csv", index=False)
monthly_kpis.to_csv("/content/drive/MyDrive/ML_PROJECT/Automated Business Reporting/data/monthly_kpis.csv", index=False)
platform_kpis.to_csv("/content/drive/MyDrive/ML_PROJECT/Automated Business Reporting/data/platform_kpis.csv", index=False)
location_kpis.to_csv("/content/drive/MyDrive/ML_PROJECT/Automated Business Reporting/data/location_kpis.csv", index=False)


## 📈 KPI Summary & Business Interpretation

- **Revenue and order volume exhibit stable growth patterns** across the observed time period, indicating consistent business performance without structural volatility.

- **Average Order Value (AOV) remains relatively stable** over time, suggesting pricing consistency and no abnormal discounting or revenue leakage.

- The **Web platform is the primary revenue and order driver**, contributing the largest share of business activity, while the **Mobile App serves as a strong secondary channel**, reflecting healthy multi-platform adoption.

- The **Marketplace**
